# Module C, Week 1: Polynomial Terms, Interactions, and Multicollinearity
### Integrated Capstone Project: Predicting Corporate Bankruptcy from Financial Ratios
**Astha Goel**

---

## Project context

This capstone predicts corporate bankruptcy from published financial ratios, running the same pipeline
separately on two datasets and comparing results rather than merging them:

| | Poland | Taiwan |
|---|---|---|
| Observations | 43,405 firm-years (2000 to 2013) | 6,819 firms (1999 to 2009) |
| Features | 64 financial ratios | 95 financial ratios |
| Bankrupt | 2,091 (4.8%) | 220 (3.2%) |

**Module B** found that no single ratio predicts bankruptcy alone (|r| < 0.04 in Poland, max 0.32 in Taiwan),
that PCA does not compress these data (PC1 captures only 15.3% and 13.4% of variance), yet a random forest
reaches ROC-AUC 0.9395 (Poland) and 0.9482 (Taiwan).

## This week

Week 1 covers polynomial terms, interaction terms, multicollinearity and the variance inflation factor, and the
handling of categorical alongside continuous features. All four turn out to be directly relevant here, because
these datasets consist almost entirely of financial ratios computed from a small set of shared underlying line
items, which is exactly the recipe for severe multicollinearity.

Four questions organize the notebook:

1. How badly collinear are these ratios, measured properly with VIF rather than eyeballed from a heatmap?
2. Do polynomial terms improve prediction, and at what point do they start overfitting?
3. Do interaction terms help, and do the two countries respond the same way?
4. Poland carries a genuine categorical feature (the forecasting horizon). Does it add anything?

## 1. Setup and loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

# shorthand used throughout: rank-based scaling, chosen in Week 4 after a six-scaler comparison
qt = lambda: QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)

In [ ]:
# ── TAIWAN ────────────────────────────────────────────────────────────────────
df_taiwan = pd.read_csv('Taiwan_data_capstone.csv')
df_taiwan.columns = df_taiwan.columns.str.strip()   # headers carry a leading space
df_taiwan['Bankrupt?'] = pd.to_numeric(df_taiwan['Bankrupt?'], errors='coerce').astype(int)

numeric_cols_taiwan = [c for c in df_taiwan.select_dtypes(include='number').columns
                       if c != 'Bankrupt?']
for col in numeric_cols_taiwan:
    df_taiwan[col] = df_taiwan[col].fillna(df_taiwan[col].median())
numeric_cols_taiwan = [c for c in numeric_cols_taiwan if df_taiwan[c].nunique() > 1]

print(f"Taiwan: {df_taiwan.shape}, {len(numeric_cols_taiwan)} usable features")
print(df_taiwan['Bankrupt?'].value_counts().to_dict())

In [ ]:
# ── POLAND ────────────────────────────────────────────────────────────────────
df_polish = pd.read_csv('Polish_dataset_capstone.csv')

column_names_64 = [
    'net_profit_to_total_assets', 'total_liabilities_to_total_assets',
    'working_capital_to_total_assets', 'current_assets_to_short_term_liabilities',
    'cash_to_operating_expenses', 'retained_earnings_to_total_assets',
    'EBIT_to_total_assets', 'book_value_of_equity_to_total_liabilities',
    'sales_to_total_assets', 'equity_to_total_assets',
    'gross_profit_to_short_term_liabilities', 'gross_profit_to_total_assets',
    'gross_profit_to_sales', 'gross_profit_to_total_assets_2',
    'working_capital_to_fixed_assets', 'receivables_to_total_assets',
    'liabilities_to_total_assets_duration', 'EBIT_to_total_assets_2',
    'sales_to_total_assets_2', 'working_capital_to_sales',
    'sales_to_inventories', 'net_profit_to_sales',
    'gross_profit_to_total_assets_3', 'current_liabilities_to_equity',
    'short_term_liabilities_to_equity', 'working_capital',
    'depreciation_to_total_assets', 'inventories_to_sales',
    'net_profit_to_inventories', 'total_liabilities_to_equity',
    'profit_on_sales_to_total_assets', 'total_liabilities_to_cash',
    'operating_expenses_to_short_term_liabilities', 'operating_expenses_to_total_liabilities',
    'profit_to_total_assets', 'net_profit_to_total_assets_2',
    'total_liabilities_to_operating_cash', 'total_liabilities_to_total_assets_rotation',
    'profit_to_long_term_liabilities', 'total_assets_to_total_liabilities',
    'short_term_liabilities_to_long_term_liabilities', 'short_term_liabilities_to_equity_2',
    'long_term_liabilities_to_equity', 'sales_to_short_term_liabilities',
    'inventory_to_receivables', 'receivables_to_sales',
    'net_profit_to_total_assets_3', 'total_costs_to_total_sales',
    'long_term_liabilities_to_total_assets', 'sales_to_equity',
    'sales_to_total_assets_3', 'liabilities_to_working_capital',
    'operating_cash_to_liabilities', 'operating_cash_to_total_liabilities',
    'operating_cash_to_sales', 'sales_to_receivables',
    'operating_expenses_to_sales', 'total_costs_to_total_assets',
    'profit_on_operating_activities', 'depreciation_to_profit',
    'total_liabilities_to_profit', 'profit_on_operating_activities_to_financial_expenses',
    'working_capital_to_liabilities', 'equity_to_fixed_assets'
]
df_polish = df_polish.rename(columns={f'A{i+1}': column_names_64[i] for i in range(64)})

# force ratio columns numeric, else select_dtypes silently skips any that arrive as strings
for c in column_names_64:
    if c in df_polish.columns:
        df_polish[c] = pd.to_numeric(df_polish[c], errors='coerce')
df_polish['class'] = pd.to_numeric(df_polish['class'], errors='coerce').astype(int)

numeric_cols_polish = [c for c in df_polish.select_dtypes(include='number').columns
                       if c not in ['class', 'year', 'id']]
for col in numeric_cols_polish:
    df_polish[col] = df_polish[col].fillna(df_polish[col].median())
numeric_cols_polish = [c for c in numeric_cols_polish if df_polish[c].nunique() > 1]

print(f"Poland: {df_polish.shape}, {len(numeric_cols_polish)} usable features")
print(df_polish['class'].value_counts().to_dict())

## 2. Multicollinearity and the variance inflation factor

A correlation heatmap shows collinearity between *pairs* of features. The variance inflation factor measures
something stricter: how well each feature can be predicted from **all the others together**. VIF for feature
*j* is 1/(1 − R²ⱼ), where R²ⱼ comes from regressing feature *j* on every other feature. Rough rule of thumb:

- VIF ≈ 1: independent of the others
- VIF > 5: moderate multicollinearity
- VIF > 10: severe; the coefficient on this feature is unstable
- VIF → ∞: the feature is an exact linear combination of others

This matters for interpreting the model rather than for predicting with it. Collinearity does not stop a model from predicting well,
but it makes individual coefficients unreliable. They can swing a lot, or flip sign, when the data change slightly, because the model cannot tell which of several interchangeable features deserves the credit.

In [ ]:
def vif_table(df, cols, name):
    X = pd.DataFrame(qt().fit_transform(df[cols]), columns=cols)
    X['const'] = 1.0   # intercept column; VIF is computed against the other predictors
    rows = [{'feature': c, 'VIF': variance_inflation_factor(X.values, i)}
            for i, c in enumerate(cols)]
    out = pd.DataFrame(rows).sort_values('VIF', ascending=False)
    print(f"\n=== {name}: variance inflation factors ===")
    print(out.to_string(index=False, float_format=lambda x: f"{x:,.1f}"))
    return out

subset_pl = ['net_profit_to_total_assets', 'total_liabilities_to_total_assets',
             'working_capital_to_total_assets', 'retained_earnings_to_total_assets',
             'EBIT_to_total_assets', 'sales_to_total_assets', 'equity_to_total_assets',
             'total_assets_to_total_liabilities', 'net_profit_to_sales',
             'total_costs_to_total_sales']

subset_tw = ['ROA(A) before interest and % after tax',
             'ROA(B) before interest and depreciation after tax',
             'ROA(C) before interest and depreciation before interest',
             'Net Income to Total Assets', 'Debt ratio %', 'Net worth/Assets',
             'Current Ratio', 'Operating Gross Margin',
             'Retained Earnings to Total Assets', 'Working Capital to Total Assets']

vif_pl = vif_table(df_polish, subset_pl, 'POLAND')
vif_tw = vif_table(df_taiwan, subset_tw, 'TAIWAN')

### Reading the VIF tables

**Taiwan contains a pair of features with VIF around 10¹⁵.** That is not just severe multicollinearity, it is
*perfect* multicollinearity: `Debt ratio %` and `Net worth/Assets` are exact complements, summing to 1 by the
accounting identity that assets equal liabilities plus equity. One is fully determined by the other, so the
regression cannot be solved and only floating-point rounding keeps the VIF finite at all. Module B
spotted this pair at a correlation of −1.00; VIF quantifies how damaging it is.

**The ROA family sits at VIF 48 to 72.** ROA(A), ROA(B), ROA(C), and Net Income to Total Assets are four
formulations of the same profitability concept. This is the redundancy behind a Module B regression return a
circular R² of 0.97, and VIF is the check that would have caught it systematically.

**Poland is collinear but not hopelessly so**, topping out at about 37. Its ratios genuinely carry more
independent information: consistent with the Module B PCA finding that PC1 captures only 15.3% of variance.

In [ ]:
# Drop one member of each redundant group and re-measure
drop = ['ROA(B) before interest and depreciation after tax',
        'ROA(C) before interest and depreciation before interest',
        'Net worth/Assets']
vif_tw_after = vif_table(df_taiwan, [c for c in subset_tw if c not in drop],
                         'TAIWAN after dropping 3 redundant features')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (tbl, name) in zip(axes, [(vif_pl, 'Poland'), (vif_tw, 'Taiwan')]):
    t = tbl.copy()
    t['VIF_plot'] = t['VIF'].clip(upper=100)        # cap so the 1e15 pair stays on-scale
    colors = ['indianred' if v > 10 else 'goldenrod' if v > 5 else 'steelblue'
              for v in t['VIF']]
    ax.barh([s[:34] for s in t['feature']][::-1], t['VIF_plot'][::-1], color=colors[::-1])
    ax.axvline(5, ls='--', c='gray', lw=1)
    ax.axvline(10, ls='--', c='indianred', lw=1)
    ax.set_xlabel('VIF (clipped at 100)')
    ax.set_title(f'{name}: red = severe (>10), amber = moderate (>5)', fontsize=9.5)
    ax.tick_params(axis='y', labelsize=7)
plt.tight_layout(); plt.show()

Dropping one member of each redundant group collapses the infinite VIF entirely and roughly halves the rest.
The practical rule I am taking from this: **before interpreting any coefficient, check VIF and prune
exact complements.** For pure prediction it matters less, which is why Module B's random forests were unaffected.

## 3. Polynomial terms

Polynomial expansion lets a linear model capture curvature by adding squared and higher-order terms. The cost is
a rapidly growing feature count and a corresponding overfitting risk, so the real test is train *and* test R²
at each degree rather than test alone.

I use the eight features most correlated with each target, computed **after winsorizing**. The order matters here:
these datasets contain values reaching 10¹⁰ (documented in Week 2), and correlations computed on raw data are
distorted by them, which selects the wrong features.

In [ ]:
def winsorize(df, cols, lower=0.01, upper=0.99):
    out = df.copy()
    for c in cols:
        lo, hi = out[c].quantile([lower, upper])
        out[c] = out[c].clip(lo, hi)
    return out

def poly_sweep(df, feats, target, name, k=8, degrees=(1, 2, 3)):
    dfw = winsorize(df, feats)
    corr = dfw[feats].corr()[target].drop(target).abs().sort_values(ascending=False)
    top = corr.head(k).index.tolist()
    X, y = dfw[top], dfw[target].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

    rows = []
    for deg in degrees:
        pipe = make_pipeline(qt(),
                             PolynomialFeatures(degree=deg, include_bias=False),
                             Ridge(alpha=1.0)).fit(Xtr, ytr)
        n_terms = pipe[1].transform(pipe[0].transform(Xtr[:3])).shape[1]
        tr, te = r2_score(ytr, pipe.predict(Xtr)), r2_score(yte, pipe.predict(Xte))
        rows.append({'degree': deg, 'terms': n_terms, 'train R2': tr,
                     'test R2': te, 'gap': tr - te})
    res = pd.DataFrame(rows)
    print(f"\n=== {name}: polynomial degree sweep, target = {target} ===")
    print(f"top {k} features: {top[:4]} ...")
    print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return res

poly_pl = poly_sweep(df_polish, numeric_cols_polish, 'sales_to_total_assets', 'POLAND')
poly_tw = poly_sweep(df_taiwan, numeric_cols_taiwan,
                     'ROA(A) before interest and % after tax', 'TAIWAN')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (res, name) in zip(axes, [(poly_pl, 'Poland'), (poly_tw, 'Taiwan')]):
    ax.plot(res['degree'], res['train R2'], marker='o', ms=7, color='indianred', label='train R2')
    ax.plot(res['degree'], res['test R2'], marker='s', ms=7, color='steelblue', label='test R2')
    ax.set_xticks(res['degree']); ax.set_xlabel('polynomial degree')
    ax.set_ylabel('R2'); ax.set_ylim(0.55, 1.02)
    ax.set_title(f'{name}', fontsize=10); ax.legend(fontsize=9)
    for _, r in res.iterrows():
        ax.annotate(f"{int(r['terms'])} terms", (r['degree'], r['test R2']),
                    textcoords='offset points', xytext=(0, -16), ha='center', fontsize=7.5)
plt.tight_layout(); plt.show()

### The two countries respond in opposite directions

**Poland improves with curvature and does not overfit.** Test R² rises from 0.659 at degree 1 to 0.863 at
degree 3, and the train-test gap stays near zero throughout. The extra 156 terms are worth it.

**Taiwan overfits.** Test R² *falls* from 0.934 to 0.831 as the degree rises, while train R² climbs to 0.967: the gap widens from 0.000 to 0.135, which is the classic sign of overfitting. Taiwan's relationship was
already essentially linear, so the added terms fit noise.

This is the first time a pattern shows up that repeats through the whole project: added flexibility helps Poland
and hurts Taiwan.

## 4. Interaction terms

Interaction terms are products of feature pairs. They let a linear model express conditional statements such as
"high leverage matters *only when* liquidity is also weak", which no additive combination can represent. Using
`interaction_only=True` isolates that effect by excluding the pure squared terms tested above.

Here I predict the actual bankruptcy target rather than a continuous ratio, and compare against a random forest, which captures interactions natively, to see how much of the gap interactions close.

In [ ]:
def interaction_test(df, feats, target, name, k=25):
    X, y = df[feats], df[target].values
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    # pick top-k once with a forest so every model below sees identical inputs
    sel = RandomForestClassifier(n_estimators=60, max_depth=10, class_weight='balanced',
                                 random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
    top = pd.Series(sel.feature_importances_, index=feats).sort_values(ascending=False).head(k).index.tolist()
    Xtr_t, Xte_t = Xtr[top], Xte[top]

    results = {}
    main = make_pipeline(qt(), LogisticRegression(C=1.0, max_iter=3000)).fit(Xtr_t, ytr)
    results['Logistic (main effects)'] = roc_auc_score(yte, main.predict_proba(Xte_t)[:, 1])

    inter = make_pipeline(qt(),
                          PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
                          LogisticRegression(C=1.0, max_iter=3000)).fit(Xtr_t, ytr)
    results['Logistic + interactions'] = roc_auc_score(yte, inter.predict_proba(Xte_t)[:, 1])
    n_terms = inter[1].transform(inter[0].transform(Xtr_t[:5])).shape[1]

    rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=5,
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr_t, ytr)
    results['Random forest'] = roc_auc_score(yte, rf.predict_proba(Xte_t)[:, 1])

    print(f"\n=== {name}: {k} features -> {n_terms} terms with interactions ===")
    for label, v in results.items():
        print(f"   {label:26s} ROC-AUC = {v:.4f}")
    gap0 = results['Random forest'] - results['Logistic (main effects)']
    gap1 = results['Random forest'] - results['Logistic + interactions']
    if gap0 > 0.005:
        print(f"   gap to forest: {gap0:.4f} -> {gap1:.4f}  ({100*(1-gap1/gap0):.0f}% closed)")
    else:
        print("   no meaningful gap - the linear model was already competitive")
    return results

int_pl = interaction_test(df_polish, numeric_cols_polish, 'class', 'POLAND')
int_tw = interaction_test(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
labels = ['Logistic\nmain effects', 'Logistic\n+ interactions', 'Random\nforest']
x = np.arange(3); w = 0.37
ax.bar(x - w/2, [int_pl[k] for k in
       ['Logistic (main effects)', 'Logistic + interactions', 'Random forest']],
       w, label='Poland', color='steelblue')
ax.bar(x + w/2, [int_tw[k] for k in
       ['Logistic (main effects)', 'Logistic + interactions', 'Random forest']],
       w, label='Taiwan', color='coral')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9.5)
ax.set_ylim(0.70, 1.0); ax.set_ylabel('Test ROC-AUC')
ax.set_title('Interaction terms help Poland; Taiwan barely moves', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
def interaction_stability(df, feats, target, name, k=25, seeds=(42, 1, 7)):
    """Repeat the main-effects vs interactions comparison across several splits."""
    rows = []
    for s in seeds:
        X, y = df[feats], df[target].values
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.2, random_state=s, stratify=y)
        sel = RandomForestClassifier(n_estimators=60, max_depth=10, class_weight='balanced',
                                     random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
        top = pd.Series(sel.feature_importances_, index=feats).sort_values(ascending=False).head(k).index.tolist()

        main = make_pipeline(qt(), LogisticRegression(C=1.0, max_iter=3000)).fit(Xtr[top], ytr)
        m = roc_auc_score(yte, main.predict_proba(Xte[top])[:, 1])
        inter = make_pipeline(qt(),
                              PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
                              LogisticRegression(C=1.0, max_iter=3000)).fit(Xtr[top], ytr)
        i_ = roc_auc_score(yte, inter.predict_proba(Xte[top])[:, 1])
        rows.append({'split seed': s, 'main effects': m, '+ interactions': i_, 'delta': i_ - m})
    res = pd.DataFrame(rows)
    print(f"\n=== {name}: is the interaction effect stable across splits? ===")
    print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"   mean delta = {res['delta'].mean():+.4f}   range = "
          f"[{res['delta'].min():+.4f}, {res['delta'].max():+.4f}]")
    return res

stab_pl = interaction_stability(df_polish, numeric_cols_polish, 'class', 'POLAND')
stab_tw = interaction_stability(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN')

### The asymmetry is real, but it is not symmetric in the way a single split suggested

Across three splits, **Poland gains consistently and substantially**, every delta is positive and large
(roughly +0.07 to +0.10). **Taiwan's delta is indistinguishable from noise**, changing sign between splits and
never exceeding about 0.02 in magnitude.

So the accurate statement is: interactions carry real signal in Poland and essentially none in Taiwan. On
any one split Taiwan may appear slightly helped or slightly hurt, and reading meaning into that direction would
be over-interpreting a single draw. I ran this check because the first split showed a small Taiwanese
movement that it would have been tempting to report as a finding.

The asymmetry still supports the hypothesis, Polish risk depends on interactions and Taiwanese risk does not, but the evidence is "Poland gains, Taiwan doesn't", not "Poland gains, Taiwan loses".

Interactions lift Poland substantially, closing most of its gap to the random forest. Taiwan barely moves.

One train/test split is not much evidence for a difference like this, so the next cell repeats the comparison
across three different splits to see whether the pattern is stable or an artifact of one draw.

## 5. Categorical and continuous features together

Almost every column in both datasets is continuous, but Poland carries one genuine categorical: the forecasting
horizon, recording how many years before the forecasting period the financial data was collected. Treating it as
a number would impose a false linear ordering, so it should be one-hot encoded.

In [ ]:
horizon = (df_polish.groupby('year')['class']
           .agg(n_firms='size', n_bankrupt='sum', bankruptcy_rate='mean'))
print("Poland: bankruptcy rate by forecasting horizon")
print(horizon.to_string(float_format=lambda x: f"{x:.4f}"))

In [ ]:
y_pl = df_polish['class'].values
X_num = df_polish[numeric_cols_polish]
X_cat = df_polish[['year']].astype(str)          # as string so it is treated as categorical

idx = np.arange(len(df_polish))
i_tr, i_te = train_test_split(idx, test_size=0.2, random_state=RANDOM_STATE, stratify=y_pl)

scores = {}

m = make_pipeline(qt(), LogisticRegression(max_iter=3000)).fit(X_num.iloc[i_tr], y_pl[i_tr])
scores['continuous only'] = roc_auc_score(y_pl[i_te], m.predict_proba(X_num.iloc[i_te])[:, 1])

m = make_pipeline(OneHotEncoder(handle_unknown='ignore'),
                  LogisticRegression(max_iter=3000)).fit(X_cat.iloc[i_tr], y_pl[i_tr])
scores['horizon (categorical) only'] = roc_auc_score(y_pl[i_te], m.predict_proba(X_cat.iloc[i_te])[:, 1])

X_both = pd.concat([X_num, X_cat], axis=1)
ct = ColumnTransformer([
    ('num', QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE), numeric_cols_polish),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['year']),
])
m = make_pipeline(ct, LogisticRegression(max_iter=3000)).fit(X_both.iloc[i_tr], y_pl[i_tr])
scores['continuous + categorical'] = roc_auc_score(y_pl[i_te], m.predict_proba(X_both.iloc[i_te])[:, 1])

print("\n=== Poland: does the categorical horizon add predictive signal? ===")
for k, v in scores.items():
    print(f"   {k:28s} ROC-AUC = {v:.4f}")

### A useful negative result

The bankruptcy rate rises monotonically with horizon, from 3.9% to 6.9%, so the variable is clearly *associated*
with the outcome. But on its own it reaches only ROC-AUC 0.543: barely above the 0.5 coin-flip baseline, and
adding it to the continuous features slightly *lowers* performance rather than raising it.

The reason is that horizon carries information about the **dataset's construction** rather than about any
individual firm. Knowing that a record came from the five-year window tells you the base rate for that stratum,
but nothing distinguishing one firm in it from another. The continuous ratios already capture whatever
firm-level distress exists.

That is worth reporting rather than quietly dropping the variable: a variable can be statistically associated with an outcome
and still be useless for prediction, and one-hot encoding it correctly is what makes that distinction visible.

## 6. Conclusions

1. **Multicollinearity here is extreme and structural.** Taiwan contains an exactly singular pair (`Debt ratio %`
   and `Net worth/Assets`, VIF ≈ 10¹⁵), and the ROA family runs at VIF 48 to 72. This is a property of
   ratio data computed from shared line items, not a data-quality defect. It means individual coefficients cannot
   be trusted without first pruning redundant features, though prediction is largely unaffected, which is why
   Module B's random forests never surfaced it.

2. **VIF is the right diagnostic for the redundancy problem I previously caught by eye.** The Module B regression
   that returned a circular R² of 0.97 involved exactly the features VIF flags here.

3. **Poland rewards nonlinearity; Taiwan does not.** Polynomial degree 3 lifts Poland's test R² from 0.659 to
   0.863 with no train-test gap, while dropping Taiwan's from 0.934 to 0.831 with a gap of 0.135: clear
   overfitting. On the bankruptcy target, interaction terms gain Poland roughly +0.07 to +0.10 ROC-AUC across
   three splits while Taiwan's change stays within noise and flips sign. The reliable claim is that interactions
   carry signal in Poland and essentially none in Taiwan.

4. **The categorical horizon is associated but not predictive.** It scores 0.543 alone and slightly hurts when
   added, because it encodes a property of how the data was collected rather than of any firm.

**Carried forward:** prune exact complements before interpreting coefficients; expect Poland to need models that
represent interactions and Taiwan to be best served by simpler additive ones; and check train-test gaps rather
than test scores alone whenever flexibility is added.